# 03 - Preprocesado

## Objetivo de este cuaderno

Preparamos los datos limpios para que los modelos los puedan usar: convertimos las variables de texto en números, separamos los datos en entrenamiento y prueba y escalamos una copia solo para la regresión lineal. Los atípicos y los nulos ya se trataron en 02_limpieza.

- **Lee:** data/processed/coches_limpio.csv
- **Guarda:** X_train.csv, X_test.csv, y_train.csv, y_test.csv, X_train_esc.csv y X_test_esc.csv en data/processed

### Importar librerías

Cargamos pandas y numpy. Las herramientas de sklearn se importan en la celda donde se usan.

In [1]:
import pandas as pd
import numpy as np

### Cargar el dataset

Leemos el dataset limpio y mostramos las primeras filas para comprobar que la carga es correcta.

In [2]:
df = pd.read_csv('../data/processed/coches_limpio.csv')
df.head()

,make,model,price,fuel,year,kms,power,doors,shift,photos,is_professional,province,publish_date
0,CITROEN,C1,6200,Gasolina,2017,50071,82,5,Manual,6,True,Tenerife,2021-01-02
1,FORD,Transit Connect,7851,Diésel,2016,103000,100,4,Manual,10,True,Barcelona,2020-12-16
2,VOLKSWAGEN,Caravelle,19426,Diésel,2014,120000,140,4,Manual,9,True,Navarra,2020-11-25
3,FORD,Transit,22850,Diésel,2017,107000,130,2,Manual,4,True,Sevilla,2021-01-12
4,PEUGEOT,3008,11490,Gasolina,2016,78665,130,5,Manual,32,True,Madrid,2021-01-08


Tamaño, tipos de cada columna y nulos.

In [3]:
print(df.shape)
df.info()

(40614, 13)
<class 'pandas.DataFrame'>
RangeIndex: 40614 entries, 0 to 40613
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   make             40614 non-null  str  
 1   model            40614 non-null  str  
 2   price            40614 non-null  int64
 3   fuel             40614 non-null  str  
 4   year             40614 non-null  int64
 5   kms              40614 non-null  int64
 6   power            40614 non-null  int64
 7   doors            40614 non-null  int64
 8   shift            40614 non-null  str  
 9   photos           40614 non-null  int64
 10  is_professional  40614 non-null  bool 
 11  province         40614 non-null  str  
 12  publish_date     40614 non-null  str  
dtypes: bool(1), int64(6), str(6)
memory usage: 3.8 MB


# Selección y transformación de variables

### Quitar publish_date y photos

La fecha de publicación no ayuda a estimar el precio (todos los anuncios se capturaron casi a la vez) y el número de fotos no es una característica del coche. Quitamos las dos columnas.

In [4]:
df = df.drop(columns=['publish_date', 'photos'])
df.shape

(40614, 11)

### Pasar el año a antigüedad

Es más fácil de interpretar la antigüedad del coche que su año. Los datos se capturaron en enero de 2021, así que la antigüedad es 2021 menos el año. Después quitamos year.

In [5]:
df['age'] = 2021 - df['year']
df = df.drop(columns='year')
df['age'].describe()

count    40614.000000
mean         6.296794
std          4.297291
min          0.000000
25%          3.000000
50%          5.000000
75%         10.000000
max         15.000000
Name: age, dtype: float64

### Vendedor profesional como 0/1

is_professional es verdadero o falso. Lo pasamos a 1 (profesional) y 0 (particular).

In [6]:
df['is_professional'] = df['is_professional'].astype(int)
df['is_professional'].value_counts()

is_professional
1    30675
0     9939
Name: count, dtype: int64

### Cambio como 0/1

Miramos qué valores tiene shift.

In [7]:
df['shift'].unique()

<StringArray>
['Manual', 'Automático']
Length: 2, dtype: str

Solo hay dos valores, así que basta una columna: 1 para automático y 0 para manual.

In [8]:
df['shift'] = df['shift'].replace({'Automático': 1, 'Manual': 0}).astype(int)
df['shift'].value_counts()

shift
0    26970
1    13644
Name: count, dtype: int64

### Provincia: península o islas

En la Pregunta 6 vimos que la provincia apenas cambia el precio en la península y que en las islas los coches son más baratos. En vez de 52 columnas, creamos una sola: isla vale 1 si el coche está en Canarias (Las Palmas y Tenerife) o en Baleares, y 0 si no. Primero comprobamos cómo se llaman esas provincias en los datos.

In [9]:
df['province'].unique()

<StringArray>
[   'Tenerife',   'Barcelona',     'Navarra',     'Sevilla',      'Madrid',
    'Baleares',    'Alicante',      'Murcia',  'Las Palmas',     'Granada',
      'Málaga',        'Lugo',      'Huelva',    'Asturias',  'Pontevedra',
     'Córdoba',    'Zaragoza',        'León',    'A Coruña',   'Cantabria',
      'Toledo',     'Vizcaya',      'Burgos',     'Almería',     'Badajoz',
      'Lleida',       'Álava',    'Valencia',       'Cádiz',    'La Rioja',
      'Girona',        'Jaén', 'Ciudad Real',       'Ávila',   'Tarragona',
      'Orense',     'Segovia',      'Cuenca',     'Cáceres',    'Albacete',
   'Castellón', 'Guadalajara',      'Huesca',      'Teruel',  'Valladolid',
   'Salamanca',   'Guipúzcoa',       'Soria',    'Palencia',      'Zamora',
     'Melilla',       'Ceuta']
Length: 52, dtype: str

In [10]:
islas = ['Las Palmas', 'Tenerife', 'Baleares']
df['isla'] = df['province'].isin(islas).astype(int)
df['isla'].value_counts()

isla
0    38644
1     1970
Name: count, dtype: int64

Quitamos province, porque ya está recogida en isla.

In [11]:
df = df.drop(columns='province')
df.head()

,make,model,price,fuel,kms,power,doors,shift,is_professional,age,isla
0,CITROEN,C1,6200,Gasolina,50071,82,5,0,1,4,1
1,FORD,Transit Connect,7851,Diésel,103000,100,4,0,1,5,0
2,VOLKSWAGEN,Caravelle,19426,Diésel,120000,140,4,0,1,7,0
3,FORD,Transit,22850,Diésel,107000,130,2,0,1,4,0
4,PEUGEOT,3008,11490,Gasolina,78665,130,5,0,1,5,0


### Combustible con one-hot

Creamos una columna 0/1 por cada combustible. El one-hot no aprende nada de los datos, solo convierte cada categoría en una columna, así que se puede hacer antes de separar train y test. Dejamos todas las categorías, igual que con la marca.

In [12]:
df = pd.get_dummies(df, columns=['fuel'], prefix='fuel', dtype=int)
df.head()

,make,model,price,kms,power,doors,shift,is_professional,age,isla,fuel_Diésel,fuel_Eléctrico,fuel_Gas licuado (GLP),fuel_Gas natural (CNG),fuel_Gasolina,fuel_Híbrido,fuel_Híbrido enchufable
0,CITROEN,C1,6200,50071,82,5,0,1,4,1,0,0,0,0,1,0,0
1,FORD,Transit Connect,7851,103000,100,4,0,1,5,0,1,0,0,0,0,0,0
2,VOLKSWAGEN,Caravelle,19426,120000,140,4,0,1,7,0,1,0,0,0,0,0,0
3,FORD,Transit,22850,107000,130,2,0,1,4,0,1,0,0,0,0,0,0
4,PEUGEOT,3008,11490,78665,130,5,0,1,5,0,0,0,0,0,1,0,0


# Separación de datos

### Variable objetivo y variables de entrada

La variable objetivo es el logaritmo del precio. El precio tiene una cola larga de coches caros; con el logaritmo la distribución es más simétrica y el error se mide en proporción al precio. X son el resto de columnas.

In [13]:
y = np.log(df['price'])
X = df.drop(columns='price')
print(X.shape, y.shape)

(40614, 16) (40614,)


### Train y test

Guardamos el 20 % de los coches para la prueba final.

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

32491 8123


# Encoding de variables categóricas

A partir de aquí, todo lo que se aprende de los datos se calcula solo con train y después se aplica a test. Así el test no influye en el preprocesado.

### Marca: las 10 más frecuentes y Other

Hay muchas marcas y algunas tienen muy pocos coches. Nos quedamos con las 10 más frecuentes en train y el resto pasa a Other.

In [15]:
top = X_train['make'].value_counts().head(10).index
top

Index(['VOLKSWAGEN', 'BMW', 'MERCEDES-BENZ', 'AUDI', 'PEUGEOT', 'FORD',
       'RENAULT', 'OPEL', 'CITROEN', 'SEAT'],
      dtype='str', name='make')

In [16]:
X_train['make'] = X_train['make'].where(X_train['make'].isin(top), 'Other')
X_test['make'] = X_test['make'].where(X_test['make'].isin(top), 'Other')
X_train['make'].value_counts()

make
Other            10174
VOLKSWAGEN        2962
BMW               2826
MERCEDES-BENZ     2639
AUDI              2486
PEUGEOT           2164
FORD              2046
RENAULT           1920
OPEL              1882
CITROEN           1787
SEAT              1605
Name: count, dtype: int64

### One-hot de la marca

Creamos una columna 0/1 por cada marca. El encoder se ajusta con train; si en test aparece una categoría que no estaba en train, se queda con todo a 0.

In [17]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[['make']])
columnas_ohe = ohe.get_feature_names_out(['make'])
columnas_ohe

array(['make_AUDI', 'make_BMW', 'make_CITROEN', 'make_FORD',
       'make_MERCEDES-BENZ', 'make_OPEL', 'make_Other', 'make_PEUGEOT',
       'make_RENAULT', 'make_SEAT', 'make_VOLKSWAGEN'], dtype=object)

In [18]:
ohe_train = pd.DataFrame(ohe.transform(X_train[['make']]), columns=columnas_ohe, index=X_train.index)
ohe_test = pd.DataFrame(ohe.transform(X_test[['make']]), columns=columnas_ohe, index=X_test.index)

In [19]:
X_train = pd.concat([X_train.drop(columns='make'), ohe_train], axis=1)
X_test = pd.concat([X_test.drop(columns='make'), ohe_test], axis=1)
print(X_train.shape, X_test.shape)

(32491, 26) (8123, 26)


### Modelo: target encoding

Hay demasiados modelos para hacer una columna por cada uno. Con target encoding cada modelo se sustituye por un número: la media del logaritmo del precio de ese modelo en train. Con smooth='auto', los modelos con pocos coches se acercan a la media general, para no fiarnos de pocas observaciones. En train, sklearn calcula ese número por partes (validación cruzada) para que cada coche no use su propio precio.

In [20]:
X_train['model'].nunique()

690

In [21]:
from sklearn.preprocessing import TargetEncoder
te = TargetEncoder(smooth='auto', target_type='continuous')
X_train['model'] = te.fit_transform(X_train[['model']], y_train)
X_test['model'] = te.transform(X_test[['model']])
X_train['model'].describe()

count    32491.000000
mean         9.488901
std          0.541192
min          6.907755
25%          9.101061
50%          9.477019
75%          9.836404
max         11.464830
Name: model, dtype: float64

### Comprobación

Todas las columnas deben ser numéricas y no debe haber nulos.

In [22]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 32491 entries, 21167 to 15795
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   model                    32491 non-null  float64
 1   kms                      32491 non-null  int64  
 2   power                    32491 non-null  int64  
 3   doors                    32491 non-null  int64  
 4   shift                    32491 non-null  int64  
 5   is_professional          32491 non-null  int64  
 6   age                      32491 non-null  int64  
 7   isla                     32491 non-null  int64  
 8   fuel_Diésel              32491 non-null  int64  
 9   fuel_Eléctrico           32491 non-null  int64  
 10  fuel_Gas licuado (GLP)   32491 non-null  int64  
 11  fuel_Gas natural (CNG)   32491 non-null  int64  
 12  fuel_Gasolina            32491 non-null  int64  
 13  fuel_Híbrido             32491 non-null  int64  
 14  fuel_Híbrido enchufable  32491 non

In [23]:
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())

0 0


# Escalado

kms, power y doors se quedan como números. La regresión lineal necesita que las variables estén en escalas parecidas; Random Forest y XGBoost no. Por eso escalamos solo unas copias, X_train_esc y X_test_esc, con StandardScaler ajustado en train.

In [24]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_esc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_esc = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
X_train_esc.describe().round(2)

,model,kms,power,doors,shift,is_professional,age,isla,fuel_Diésel,fuel_Eléctrico,...,make_BMW,make_CITROEN,make_FORD,make_MERCEDES-BENZ,make_OPEL,make_Other,make_PEUGEOT,make_RENAULT,make_SEAT,make_VOLKSWAGEN
count,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,...,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00,32491.00
mean,0.00,0.00,0.00,-0.00,0.00,-0.00,-0.00,0.00,0.00,-0.00,...,-0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-4.77,-1.21,-1.67,-3.21,-0.71,-1.75,-1.47,-0.23,-1.30,-0.12,...,-0.31,-0.24,-0.26,-0.30,-0.25,-0.68,-0.27,-0.25,-0.23,-0.32
25%,-0.72,-0.84,-0.53,-0.72,-0.71,0.57,-0.77,-0.23,-1.30,-0.12,...,-0.31,-0.24,-0.26,-0.30,-0.25,-0.68,-0.27,-0.25,-0.23,-0.32
50%,-0.02,-0.15,-0.22,0.52,-0.71,0.57,-0.30,-0.23,0.77,-0.12,...,-0.31,-0.24,-0.26,-0.30,-0.25,-0.68,-0.27,-0.25,-0.23,-0.32
75%,0.64,0.59,0.21,0.52,1.40,0.57,0.86,-0.23,0.77,-0.12,...,-0.31,-0.24,-0.26,-0.30,-0.25,1.48,-0.27,-0.25,-0.23,-0.32
max,3.65,10.79,7.40,0.52,1.40,0.57,2.02,4.42,0.77,8.46,...,3.24,4.15,3.86,3.36,4.03,1.48,3.74,3.99,4.39,3.16


No usamos PCA: tenemos pocas variables y queremos que el modelo se pueda explicar.

# Guardar los conjuntos

In [25]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
X_train_esc.to_csv('../data/processed/X_train_esc.csv', index=False)
X_test_esc.to_csv('../data/processed/X_test_esc.csv', index=False)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(32491, 26) (8123, 26) (32491,) (8123,)


## Resumen

- Partimos de 40.614 coches y 13 columnas.
- Quitamos publish_date y photos, y cambiamos year por age (2021 menos el año).
- is_professional y shift pasan a 0/1 (automático = 1), y province se sustituye por isla (Las Palmas, Tenerife y Baleares = 1).
- La variable objetivo es el logaritmo de price.
- Train tiene 32.491 coches y test 8.123 (80/20).
- Combustible con one-hot antes de separar (7 columnas).
- Marca: las 10 más frecuentes en train y el resto en Other, con one-hot ajustado en train (11 columnas).
- Modelo (690 modelos distintos en train) con target encoding ajustado solo en train.
- Quedan 26 columnas numéricas y sin nulos.
- X_train_esc y X_test_esc son copias escaladas con StandardScaler, solo para la regresión lineal.